# Movie Recap AI - Google Drive Edition
Welcome to the high-performance Colab version of Movie Recap AI! 

**CRITICAL STEP**: Before running anything, make sure you have a GPU enabled to speed up the Audio Transcription (Whisper) model.
1. Go to the menu bar at the top: **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU**.
3. Click **Save**.

*Note: This version saves all your files permanently to your Google Drive, while using Colab's high-speed local disk for temporary processing.*

In [ ]:
# Step 1: Mount Google Drive & Setup Code (Auto-Git Sync)
from google.colab import drive
import os

print("[*] Mounting Google Drive...")
drive.mount("/content/drive")

drive_path = "/content/drive/MyDrive/AI-Movie-Translate"

if not os.path.exists(drive_path):
    print("[*] First time setup: Cloning repository into Google Drive...")
    %cd /content/drive/MyDrive
    !git clone https://github.com/paipai1999/AI-Movie-Translate.git
else:
    print("[*] Existing project found in Google Drive! Syncing latest updates from GitHub...")
    %cd {drive_path}
    !git reset --hard HEAD
    !git pull origin main

%cd {drive_path}

# Ensure local NVMe temp directory is ready
os.makedirs("/content/temp", exist_ok=True)

print("[*] Installing required Python libraries & FFmpeg...")
!pip install -q -r requirements.txt
!sudo apt update -qq
!sudo apt install -y -qq ffmpeg fonts-noto-core fonts-noto-cjk fonts-sil-padauk
print("✅ Git & Drive Setup Complete! Ready for Step 2.")

In [ ]:
import json
import os

# Step 2: Configure your API Key and High-Speed Settings
# Paste your official Google AI Studio Gemini API Key 
GEMINI_API_KEYS = [
    "YOUR_GEMINI_API_KEY_HERE"
]

# Ensure high-speed local storage is used for temporary files
temp_storage_path = "/content/temp"
os.makedirs(temp_storage_path, exist_ok=True)

config_data = {
    "gemini": {
        "enabled": True,
        "api_keys": GEMINI_API_KEYS,
        "model": "gemini-3.1-flash-lite",
        "daily_limit_per_key": 500,
        "model_limits": {
            "gemini-3.1-flash-lite": 500,
            "gemini-3.5-flash-lite": 500,
            "gemini-3.7-flash": 20,
            "gemini-3.6-flash": 20,
            "gemini-3.5-flash": 20,
            "gemini-3-flash": 20,
            "gemini-2.5-flash": 20,
            "gemini-2.5-flash-lite": 20
        },
        "models": {
            "heavy": "gemini-3.7-flash",
            "workhorse": "gemini-3.1-flash-lite",
            "polish": "gemini-3.5-flash-lite"
        }
    },
    "pipeline": {
        "language": "burmese",
        "whisper_model": "small",
        "parallel_processing": True,
        "use_demucs": True
    },
    "paths": {
        "temp_dir": temp_storage_path,
        "output_dir": "outputs"
    },
    "voice": {
        "enabled": True,
        "voice_mode": "dynamic",
        "tts_voice_mm": "my-MM-ThihaNeural",
        "tts_voice_en": "en-US-GuyNeural",
        "tts_voice": "my-MM-ThihaNeural",
        "tts_rate_mm": "+8%",
        "tts_rate_en": "+15%"
    },
    "subtitle_overlay": {
        "enabled": True,
        "font_name": "Padauk",
        "font_size": 40,
        "bold": True,
        "border_style": 3,
        "outline_width": 3,
        "margin_bottom": 50,
        "max_chars_per_line": 28
    }
}

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=4)

print(f"✅ config.json created successfully in Google Drive!")
print(f"🚀 Workhorse Model: gemini-3.1-flash-lite (500 RPD) | Storage: {temp_storage_path}")

In [ ]:
# Step 3: Run the AI Pipeline (Command Line Mode)
VIDEO_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID" # <--- Paste your URL here

# Start the process!
!python main.py -i "$VIDEO_URL" --blocks 5

In [ ]:
# (OPTIONAL) Step 4: Run the Web UI Dashboard in Colab
# If you prefer the beautiful Web UI instead of the command line in Step 3, run this cell!
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import os
import subprocess
import time
import sys

# Ensure we are in the Google Drive directory
drive_path = '/content/drive/MyDrive/AI-Movie-Translate'
if os.path.exists(drive_path):
    os.chdir(drive_path)
else:
    print("❌ ERROR: Cannot find project in Google Drive!")
    print("💡 PLEASE RUN **STEP 1** FIRST.")
    sys.exit(1)

print("===========================================================")
print("🧹 Cleaning up old processes...")
!pkill -f "python web_ui.py" || true
!pkill -f "cloudflared" || true

print("🚀 Starting Web UI...")
# Start Flask and capture any crash logs
flask_proc = subprocess.Popen(
    ['python', 'web_ui.py'], 
    stdout=open('/content/web_ui.log', 'w'), 
    stderr=subprocess.STDOUT
)
time.sleep(5)

# Check if Flask crashed immediately
if flask_proc.poll() is not None:
    print("❌ ERROR: Web UI failed to start! Check the logs below:")
    with open('/content/web_ui.log', 'r') as f:
        print(f.read())
else:
    print("🌐 Creating secure Cloudflare tunnel...")
    print("👇 LOOK FOR THE LINK ENDING WITH '.trycloudflare.com' IN THE LOGS BELOW 👇")
    print("===========================================================\n")
    !./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:5000


In [ ]:
# Step 5: (Optional) Your videos are already saved to Google Drive!
# You can just open your Google Drive app and look in AI-Movie-Translate/outputs
print("✅ Your finished videos are automatically saved in Google Drive > AI-Movie-Translate > outputs")